# Ứng Dụng 6 Mô Hình Học Sâu Tăng Cường (DRL) Từ FinRL Vào Game Tic-Tac-Toe

> ***"FinRL provides a unified framework for financial reinforcement learning, integrating state-of-the-art DRL algorithms from Stable-Baselines3 to solve complex sequential decision-making problems."***

---

## 🎯 Mục Tiêu Dự Án
Tái hiện và chuẩn hóa bộ khung **DRL Agent của thư viện FinRL** (`finrl/agents/stablebaselines3/models.py`) để áp dụng huấn luyện cho bài toán môi trường Game Tic-Tac-Toe. Toàn bộ notebook được tổ chức thành các cell độc lập theo đúng quy chuẩn:

1. **Code gốc khởi tạo từng loại mô hình**: Áp dụng đúng siêu tham số (`MODEL_KWARGS`) và cấu trúc kiến trúc từ FinRL config.
2. **Thư mục lưu trữ Checkpoint**: Toàn bộ mô hình đã huấn luyện được lưu tại `practice/TicTacToe_framework/files/FinRL_Checkpoints`.
3. **Tách biệt rõ ràng từng cell**:
   - **Cell Khởi tạo** (Model Initialization & Hyperparameters).
   - **Cell Huấn luyện** (Training Execution & Callback Logging).
   - **Cell Trực quan hóa** (Đường cong Loss, Q-learning Value tại các trạng thái $t$, và Rewards).
   - **Cell Kiểm thử / Chạy mô hình** (Inference & Thi đấu đối kháng).
4. **Tập trung vào 6 mô hình DRL nòng cốt của FinRL trong Stable-Baselines3**:
   - **DQN** (Deep Q-Network - Value-Based Q-Learning)
   - **PPO** (Proximal Policy Optimization - On-Policy Actor-Critic)
   - **A2C** (Advantage Actor-Critic - Synchronous Policy Gradient)
   - **SAC** (Soft Actor-Critic - Maximum Entropy Continuous Control)
   - **TD3** (Twin Delayed DDPG - Continuous Actor-Critic với Twin Critics)
   - **DDPG** (Deep Deterministic Policy Gradient - Continuous Control)

## 1. Khởi Tạo Môi Trường & Thư Viện Chung

In [ ]:
# Cài đặt và nhập các thư viện cốt lõi
import os
import sys
import time
import random
from copy import deepcopy
import numpy as np
import matplotlib.pyplot as plt

# Nhập thư viện Gymnasium chuẩn cho RL
try:
    import gymnasium as gym
    from gymnasium import spaces
except ImportError:
    import gym
    from gym import spaces

# Nhập các thuật toán DRL Stable-Baselines3 từ cấu hình FinRL
from stable_baselines3 import A2C, PPO, DQN, SAC, TD3, DDPG
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.noise import NormalActionNoise, OrnsteinUhlenbeckActionNoise
from stable_baselines3.common.vec_env import DummyVecEnv

# Thiết lập đường dẫn lưu trữ Checkpoint theo yêu cầu
CHECKPOINT_DIR = os.path.join("practice", "TicTacToe_framework", "files", "FinRL_Checkpoints")
if not os.path.exists(CHECKPOINT_DIR):
    # Fallback nếu đang chạy từ thư mục con
    if os.path.exists(os.path.join("files")):
        CHECKPOINT_DIR = os.path.join("files", "FinRL_Checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f"✅ Thư viện sẵn sàng!")
print(f"📁 Thư mục lưu checkpoint mô hình: {os.path.abspath(CHECKPOINT_DIR)}")

### 1.1. Thiết lập Đối thủ Cờ: Minimax AI & Random AI
- **Minimax AI**: Đối thủ chơi chuẩn xác không mắc sai lầm, đóng vai trò thước đo chuẩn mực (Benchmark) để kiểm tra chất lượng của các Agent DRL.
- **Random AI**: Đối thủ ngẫu nhiên, giúp mô hình tích lũy điểm thưởng và học cơ bản các thế thắng trong giai đoạn đầu.

In [ ]:
WIN_COMBINATIONS = [
    [0, 1, 2], [3, 4, 5], [6, 7, 8], # Ngang
    [0, 3, 6], [1, 4, 7], [2, 5, 8], # Dọc
    [0, 4, 8], [2, 4, 6]             # Chéo
]

def check_winner(board):
    """Kiểm tra trạng thái thắng/thua/hòa của bàn cờ (1: X thắng, -1: O thắng, 0: hòa/chưa xong)."""
    for line in WIN_COMBINATIONS:
        s = board[line[0]] + board[line[1]] + board[line[2]]
        if s == 3:
            return 1 # X thắng
        elif s == -3:
            return -1 # O thắng
    if 0 not in board:
        return 0 # Hòa
    return None # Đang chơi

def minimax_search(board, depth, is_maximizing, player_mark):
    """Thuật toán Minimax tìm nước đi tối ưu cho đối thủ."""
    winner = check_winner(board)
    if winner is not None:
        return winner * player_mark, None

    best_move = None
    if is_maximizing:
        max_eval = -float('inf')
        for i in range(9):
            if board[i] == 0:
                board[i] = player_mark
                eval_score, _ = minimax_search(board, depth + 1, False, player_mark)
                board[i] = 0
                if eval_score > max_eval:
                    max_eval = eval_score
                    best_move = i
        return max_eval, best_move
    else:
        min_eval = float('inf')
        opp_mark = -player_mark
        for i in range(9):
            if board[i] == 0:
                board[i] = opp_mark
                eval_score, _ = minimax_search(board, depth + 1, True, player_mark)
                board[i] = 0
                if eval_score < min_eval:
                    min_eval = eval_score
                    best_move = i
        return min_eval, best_move

def get_opponent_move(board, opp_type="minimax", opp_mark=-1):
    """Lấy nước đi của đối thủ (Minimax hoặc Random)."""
    empty_cells = [i for i in range(9) if board[i] == 0]
    if not empty_cells:
        return None
    if opp_type == "random":
        return random.choice(empty_cells)
    # Mặc định minimax: với 20% ngẫu nhiên để agent học nhiều thế cờ đa dạng
    if random.random() < 0.2:
        return random.choice(empty_cells)
    _, best_move = minimax_search(board, 0, True, opp_mark)
    return best_move if best_move is not None else random.choice(empty_cells)

print("✅ Đối thủ Minimax AI & Random AI đã sẵn sàng!")

### 1.2. Môi Trường Game Tic-Tac-Toe Tương Thích Gym / Gymnasium
Để tương thích với toàn bộ 6 mô hình của FinRL, chúng ta xây dựng 2 chuẩn môi trường:
1. **`TicTacToeDiscreteEnv`**: Dành cho các mô hình hành động rời rạc (**DQN, PPO, A2C**), `action_space = spaces.Discrete(9)`.
2. **`TicTacToeContinuousEnv`**: Dành cho các mô hình điều khiển liên tục (**SAC, TD3, DDPG**), `action_space = spaces.Box(low=-1, high=1, shape=(9,))`. Mô hình xuất ra vector 9 chiều thể hiện điểm ưu tiên cho 9 ô, và hệ thống sẽ chọn ô hợp lệ có điểm cao nhất (`argmax`).

In [ ]:
class BaseTicTacToeEnv(gym.Env):
    """Môi trường cơ sở cho Tic-Tac-Toe tuân thủ chuẩn Gym/Gymnasium."""
    metadata = {"render_modes": ["human"]}

    def __init__(self, opponent_type="minimax"):
        super().__init__()
        self.opponent_type = opponent_type
        # Trạng thái bàn cờ 9 ô: 1 (Agent X), -1 (Đối thủ O), 0 (Ô trống)
        self.observation_space = spaces.Box(low=-1.0, high=1.0, shape=(9,), dtype=np.float32)
        self.board = np.zeros(9, dtype=np.int8)
        self.current_step = 0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.board = np.zeros(9, dtype=np.int8)
        self.current_step = 0
        # 50% xác suất đối thủ đi trước để agent học cả 2 thế đi trước (X) và đi sau (O)
        if random.random() < 0.5:
            opp_first_move = get_opponent_move(self.board, opp_type="random", opp_mark=-1)
            if opp_first_move is not None:
                self.board[opp_first_move] = -1
        obs = self.board.astype(np.float32)
        return obs, {}

    def render(self, mode="human"):
        symbols = {1: ' X ', -1: ' O ', 0: ' . '}
        print("\n-------------")
        for row in range(3):
            print("|" + "|".join([symbols[self.board[row*3 + col]] for col in range(3)]) + "|")
            print("-------------")

class TicTacToeDiscreteEnv(BaseTicTacToeEnv):
    """Môi trường không gian hành động rời rạc (DQN, PPO, A2C)."""
    def __init__(self, opponent_type="minimax"):
        super().__init__(opponent_type=opponent_type)
        self.action_space = spaces.Discrete(9)

    def step(self, action):
        action = int(action)
        self.current_step += 1

        # 1. Phạt nếu đánh vào ô không hợp lệ (đã có quân)
        if self.board[action] != 0:
            reward = -1.0 # Phạt nặng để ép mô hình học luật ô trống
            terminated = True
            truncated = False
            return self.board.astype(np.float32), reward, terminated, truncated, {"result": "invalid"}

        # 2. Agent X đánh vào ô action
        self.board[action] = 1
        status = check_winner(self.board)
        if status == 1:
            return self.board.astype(np.float32), 1.0, True, False, {"result": "win"}
        elif status == 0:
            return self.board.astype(np.float32), 0.2, True, False, {"result": "draw"}

        # 3. Lượt đối thủ O phản công
        opp_move = get_opponent_move(self.board, opp_type=self.opponent_type, opp_mark=-1)
        if opp_move is not None:
            self.board[opp_move] = -1

        # 4. Kiểm tra kết quả sau lượt đối thủ
        status = check_winner(self.board)
        if status == -1:
            return self.board.astype(np.float32), -1.0, True, False, {"result": "loss"}
        elif status == 0:
            return self.board.astype(np.float32), 0.2, True, False, {"result": "draw"}

        # Ván cờ tiếp diễn, thưởng nhỏ khuyến khích đi nước hợp lệ
        return self.board.astype(np.float32), 0.05, False, False, {"result": "ongoing"}

class TicTacToeContinuousEnv(BaseTicTacToeEnv):
    """Môi trường không gian hành động liên tục (SAC, TD3, DDPG)."""
    def __init__(self, opponent_type="minimax"):
        super().__init__(opponent_type=opponent_type)
        # Vector hành động liên tục 9 chiều [-1, 1]
        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(9,), dtype=np.float32)

    def step(self, action):
        self.current_step += 1
        action = np.array(action).flatten()
        
        # Chuyển đổi continuous logits sang nước đi rời rạc hợp lệ có điểm số cao nhất
        empty_cells = [i for i in range(9) if self.board[i] == 0]
        if not empty_cells:
            return self.board.astype(np.float32), 0.0, True, False, {"result": "draw"}
        
        # Chọn ô trống có giá trị hành động liên tục lớn nhất
        discrete_action = max(empty_cells, key=lambda c: action[c])
        
        self.board[discrete_action] = 1
        status = check_winner(self.board)
        if status == 1:
            return self.board.astype(np.float32), 1.0, True, False, {"result": "win"}
        elif status == 0:
            return self.board.astype(np.float32), 0.2, True, False, {"result": "draw"}

        # Lượt đối thủ phản công
        opp_move = get_opponent_move(self.board, opp_type=self.opponent_type, opp_mark=-1)
        if opp_move is not None:
            self.board[opp_move] = -1

        status = check_winner(self.board)
        if status == -1:
            return self.board.astype(np.float32), -1.0, True, False, {"result": "loss"}
        elif status == 0:
            return self.board.astype(np.float32), 0.2, True, False, {"result": "draw"}

        return self.board.astype(np.float32), 0.05, False, False, {"result": "ongoing"}

print("✅ Môi trường TicTacToeDiscreteEnv và TicTacToeContinuousEnv đã sẵn sàng!")

### 1.3. Callback Ghi Nhận Metric Huấn Luyện (Loss & Q-Learning Value)
Kế thừa từ `BaseCallback` của Stable-Baselines3 (tương tự `TensorboardCallback` trong `finrl/agents/stablebaselines3/models.py`), callback này sẽ tự động ghi nhận:
- **Loss qua các bước cập nhật**: Critic Loss / Value Loss, Actor Loss / Policy Loss.
- **Giá trị Q-learning / Value Function tại các trạng thái $t$**: Đánh giá giá trị kỳ vọng $Q(s_t, a_t)$ hoặc $V(s_t)$ trên tập trạng thái thế cờ mẫu để vẽ đường cong học tập.

In [ ]:
class FinRLGameMetricsCallback(BaseCallback):
    """
    Callback thu thập Loss, Q-values và Rewards trong suốt quá trình huấn luyện.
    """
    def __init__(self, eval_freq=100, verbose=0):
        super().__init__(verbose)
        self.eval_freq = eval_freq
        self.timesteps = []
        self.losses = []
        self.q_values = []
        self.ep_rewards = []
        
        # Tập trạng thái mẫu đại diện để theo dõi giá trị Q-value ở các thế cờ t:
        # 1. Bàn cờ trống t=0; 2. Thế cờ mở đầu t=1; 3. Thế cờ trung cuộc t=3
        self.sample_states = np.array([
            [0, 0, 0, 0, 0, 0, 0, 0, 0],       # t=0: Trống
            [0, 0, 0, 0, 1, 0, 0, 0, 0],       # t=1: X chiếm tâm
            [1, 0, 0, 0, -1, 0, 0, 0, 1],      # t=3: X ở 2 góc, O ở tâm
        ], dtype=np.float32)

    def _on_step(self) -> bool:
        # Thu thập phần thưởng episode
        if "rewards" in self.locals:
            self.ep_rewards.append(float(np.mean(self.locals["rewards"])))

        # Thu thập Loss và Q-values định kỳ
        if self.n_calls % self.eval_freq == 0:
            self.timesteps.append(self.num_timesteps)
            
            # 1. Trích xuất Loss từ Logger của SB3
            step_loss = 0.0
            if hasattr(self.model, "logger") and self.model.logger:
                for key in ["train/loss", "train/critic_loss", "train/value_loss", "train/policy_loss"]:
                    if key in self.model.logger.name_to_value:
                        step_loss = self.model.logger.name_to_value[key]
                        break
            self.losses.append(step_loss)

            # 2. Đo lường giá trị Q-learning / Value Function tại các trạng thái mẫu t
            q_mean = 0.0
            try:
                import torch
                obs_tensor = torch.as_tensor(self.sample_states, device=self.model.device)
                with torch.no_grad():
                    # Với DQN: lấy Max Q(s, a)
                    if hasattr(self.model, "q_net"):
                        q_out = self.model.q_net(obs_tensor)
                        q_mean = q_out.max(dim=1).values.mean().item()
                    # Với Critic của SAC, TD3, DDPG
                    elif hasattr(self.model, "critic") and hasattr(self.model, "actor"):
                        actions = self.model.actor(obs_tensor)
                        q_out = self.model.critic(obs_tensor, actions)[0]
                        q_mean = q_out.mean().item()
                    # Với Value Network của PPO, A2C
                    elif hasattr(self.model, "policy") and hasattr(self.model.policy, "predict_values"):
                        val_out = self.model.policy.predict_values(obs_tensor)
                        q_mean = val_out.mean().item()
            except Exception:
                q_mean = float(np.mean(self.ep_rewards[-10:])) if self.ep_rewards else 0.0
            
            self.q_values.append(q_mean)
        return True

def plot_model_curves(cb, model_name):
    """Vẽ 3 đường cong học tập: Loss, Q-Learning Value và Cumulative Rewards."""
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    fig.suptitle(f"Đường Cong Huấn Luyện Mô Hình: {model_name.upper()} (FinRL)", fontsize=14, fontweight='bold')

    # 1. Đường cong Loss
    axes[0].plot(cb.timesteps, cb.losses, color='#ef4444', lw=2, label='Training Loss')
    axes[0].set_title("1. Training Loss Curve")
    axes[0].set_xlabel("Timesteps")
    axes[0].set_ylabel("Loss Value")
    axes[0].grid(True, linestyle="--", alpha=0.5)
    axes[0].legend()

    # 2. Đường cong Q-learning / Value Function ở các trạng thái t
    axes[1].plot(cb.timesteps, cb.q_values, color='#0284c7', lw=2, label='Expected Q / Value')
    axes[1].set_title("2. Q-Learning Value tại các trạng thái t")
    axes[1].set_xlabel("Timesteps")
    axes[1].set_ylabel("Mean Q / Value")
    axes[1].grid(True, linestyle="--", alpha=0.5)
    axes[1].legend()

    # 3. Đường cong Phần thưởng (Rewards)
    if cb.ep_rewards:
        # Rolling mean của rewards
        window = max(1, len(cb.ep_rewards) // 30)
        smoothed = np.convolve(cb.ep_rewards, np.ones(window)/window, mode='valid')
        axes[2].plot(smoothed, color='#16a34a', lw=2, label=f'Reward (MA {window})')
    axes[2].set_title("3. Episode Rewards Progression")
    axes[2].set_xlabel("Step Intervals")
    axes[2].set_ylabel("Average Reward")
    axes[2].grid(True, linestyle="--", alpha=0.5)
    axes[2].legend()

    plt.tight_layout()
    plt.show()

def evaluate_trained_agent(model, env, num_games=10, is_continuous=False):
    """Kiểm thử mô hình đã huấn luyện thi đấu 10 ván và in kết quả."""
    wins, losses, draws = 0, 0, 0
    print(f"\n🎮 BẮT ĐẦU KIỂM THỬ {num_games} VÁN ĐỐI ĐẦU MINIMAX AI:")
    
    for g in range(num_games):
        obs, _ = env.reset()
        done = False
        while not done:
            if is_continuous:
                action, _ = model.predict(obs, deterministic=True)
                obs, reward, terminated, truncated, info = env.step(action)
            else:
                action, _ = model.predict(obs, deterministic=True)
                obs, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            
        res = info.get("result", "draw")
        if res == "win":
            wins += 1
            tag = "🏆 THẮNG"
        elif res == "loss":
            losses += 1
            tag = "❌ THUA"
        elif res == "invalid":
            losses += 1
            tag = "⚠️ PHẠM QUY"
        else:
            draws += 1
            tag = "🤝 HÒA"
        print(f"Ván {g+1:02d}: {tag} (Phần thưởng: {reward:+.2f})")
        
    print(f"\n📊 TỔNG KẾT: Thắng: {wins}/{num_games} | Hòa: {draws}/{num_games} | Thua: {losses}/{num_games}")
    print(f"Tỷ lệ bất bại: {(wins + draws) / num_games * 100:.1f}%")

print("✅ Callback và hàm trực quan hóa đã hoàn thành!")

## 2. Mô Hình 1: DQN (Deep Q-Network - Value-Based Q-Learning)

- **Đặc tính**: Thuật toán học giá trị ngoại vi (Off-Policy Value-Based) cốt lõi của RL.
- **Công thức tối ưu hóa hàm Q**: 
  $$\mathcal{L}(\theta) = \mathbb{E} \left[ \left( r + \gamma \max_{a'} Q(s', a'; \theta^-) - Q(s, a; \theta) \right)^2 \right]$$

In [ ]:
# CELL 1: KHỞI TẠO MÔ HÌNH DQN (CODE GỐC FINRL CONFIG)
env_dqn = DummyVecEnv([lambda: TicTacToeDiscreteEnv(opponent_type="minimax")])

# Cấu hình siêu tham số phỏng theo FinRL DRLAgent config
DQN_KWARGS = {
    "learning_rate": 0.0005,
    "buffer_size": 20000,
    "learning_starts": 200,
    "batch_size": 64,
    "gamma": 0.99,
    "exploration_fraction": 0.4,
    "exploration_initial_eps": 1.0,
    "exploration_final_eps": 0.05,
    "target_update_interval": 250,
    "verbose": 0,
}

model_dqn = DQN(
    policy="MlpPolicy",
    env=env_dqn,
    seed=42,
    policy_kwargs=dict(net_arch=[128, 64]),
    **DQN_KWARGS
)

print("✅ Mô hình DQN đã được khởi tạo thành công với cấu hình FinRL!")
print(model_dqn.policy)

In [ ]:
# CELL 2: HUẤN LUYỆN DQN VÀ LƯU CHECKPOINT
DQN_TIMESTEPS = 5000
dqn_callback = FinRLGameMetricsCallback(eval_freq=100)

print(f"🚀 Bắt đầu huấn luyện DQN ({DQN_TIMESTEPS} timesteps)...")
start_time = time.time()
model_dqn.learn(total_timesteps=DQN_TIMESTEPS, callback=dqn_callback)
print(f"⏱️ Huấn luyện xong sau: {time.time() - start_time:.2f}s")

# Lưu Checkpoint vào thư mục yêu cầu
dqn_save_path = os.path.join(CHECKPOINT_DIR, "dqn_ttt.zip")
model_dqn.save(dqn_save_path)
print(f"💾 Đã lưu Checkpoint DQN thành công tại: {dqn_save_path}")

In [ ]:
# CELL 3: VẼ ĐƯỜNG CONG HUẤN LUYỆN LOSS VÀ Q-LEARNING VALUE CỦA DQN
plot_model_curves(dqn_callback, model_name="DQN")

In [ ]:
# CELL 4: CHẠY THỬ NGHIỆM MÔ HÌNH DQN ĐÃ HUẤN LUYỆN
eval_env_dqn = TicTacToeDiscreteEnv(opponent_type="minimax")
loaded_dqn = DQN.load(dqn_save_path, env=eval_env_dqn)
evaluate_trained_agent(loaded_dqn, eval_env_dqn, num_games=10, is_continuous=False)

## 3. Mô Hình 2: PPO (Proximal Policy Optimization - On-Policy Actor-Critic)

- **Đặc tính**: Mô hình On-Policy chủ lực được FinRL khuyến nghị nhiều nhất nhờ độ ổn định vượt bậc.
- **Cơ chế Clipped Objective**: Giới hạn tỉ lệ cập nhật chính sách $\text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon)$ để ngăn ngừa việc sụp đổ hiệu năng chính sách.

In [ ]:
# CELL 1: KHỞI TẠO MÔ HÌNH PPO (CODE GỐC FINRL PPO_PARAMS)
env_ppo = DummyVecEnv([lambda: TicTacToeDiscreteEnv(opponent_type="minimax")])

# Siêu tham số chuẩn từ finrl/config.py: PPO_PARAMS
PPO_KWARGS = {
    "n_steps": 512,
    "ent_coef": 0.01,
    "learning_rate": 0.00025,
    "batch_size": 64,
    "gamma": 0.99,
    "clip_range": 0.2,
    "verbose": 0,
}

model_ppo = PPO(
    policy="MlpPolicy",
    env=env_ppo,
    seed=42,
    policy_kwargs=dict(net_arch=dict(pi=[128, 64], vf=[128, 64])),
    **PPO_KWARGS
)

print("✅ Mô hình PPO đã được khởi tạo thành công với cấu hình FinRL!")
print(model_ppo.policy)

In [ ]:
# CELL 2: HUẤN LUYỆN PPO VÀ LƯU CHECKPOINT
PPO_TIMESTEPS = 6000
ppo_callback = FinRLGameMetricsCallback(eval_freq=100)

print(f"🚀 Bắt đầu huấn luyện PPO ({PPO_TIMESTEPS} timesteps)...")
start_time = time.time()
model_ppo.learn(total_timesteps=PPO_TIMESTEPS, callback=ppo_callback)
print(f"⏱️ Huấn luyện xong sau: {time.time() - start_time:.2f}s")

# Lưu Checkpoint vào thư mục yêu cầu
ppo_save_path = os.path.join(CHECKPOINT_DIR, "ppo_ttt.zip")
model_ppo.save(ppo_save_path)
print(f"💾 Đã lưu Checkpoint PPO thành công tại: {ppo_save_path}")

In [ ]:
# CELL 3: VẼ ĐƯỜNG CONG HUẤN LUYỆN LOSS VÀ VALUE CURVE CỦA PPO
plot_model_curves(ppo_callback, model_name="PPO")

In [ ]:
# CELL 4: CHẠY THỬ NGHIỆM MÔ HÌNH PPO ĐÃ HUẤN LUYỆN
eval_env_ppo = TicTacToeDiscreteEnv(opponent_type="minimax")
loaded_ppo = PPO.load(ppo_save_path, env=eval_env_ppo)
evaluate_trained_agent(loaded_ppo, eval_env_ppo, num_games=10, is_continuous=False)

## 4. Mô Hình 3: A2C (Advantage Actor-Critic - Synchronous Policy Gradient)

- **Đặc tính**: Phiên bản đồng bộ của A3C, cập nhật tham số song song với độ trễ tối thiểu.
- **Hàm Lợi thế (Advantage Function)**: $A(s, a) = Q(s, a) - V(s)$ giúp giảm mạnh phương sai (variance) của gradient chính sách.

In [ ]:
# CELL 1: KHỞI TẠO MÔ HÌNH A2C (CODE GỐC FINRL A2C_PARAMS)
env_a2c = DummyVecEnv([lambda: TicTacToeDiscreteEnv(opponent_type="minimax")])

# Siêu tham số chuẩn từ finrl/config.py: A2C_PARAMS
A2C_KWARGS = {
    "n_steps": 5,
    "ent_coef": 0.01,
    "learning_rate": 0.0007,
    "gamma": 0.99,
    "verbose": 0,
}

model_a2c = A2C(
    policy="MlpPolicy",
    env=env_a2c,
    seed=42,
    policy_kwargs=dict(net_arch=dict(pi=[128, 64], vf=[128, 64])),
    **A2C_KWARGS
)

print("✅ Mô hình A2C đã được khởi tạo thành công với cấu hình FinRL!")
print(model_a2c.policy)

In [ ]:
# CELL 2: HUẤN LUYỆN A2C VÀ LƯU CHECKPOINT
A2C_TIMESTEPS = 6000
a2c_callback = FinRLGameMetricsCallback(eval_freq=100)

print(f"🚀 Bắt đầu huấn luyện A2C ({A2C_TIMESTEPS} timesteps)...")
start_time = time.time()
model_a2c.learn(total_timesteps=A2C_TIMESTEPS, callback=a2c_callback)
print(f"⏱️ Huấn luyện xong sau: {time.time() - start_time:.2f}s")

# Lưu Checkpoint vào thư mục yêu cầu
a2c_save_path = os.path.join(CHECKPOINT_DIR, "a2c_ttt.zip")
model_a2c.save(a2c_save_path)
print(f"💾 Đã lưu Checkpoint A2C thành công tại: {a2c_save_path}")

In [ ]:
# CELL 3: VẼ ĐƯỜNG CONG HUẤN LUYỆN LOSS VÀ VALUE CURVE CỦA A2C
plot_model_curves(a2c_callback, model_name="A2C")

In [ ]:
# CELL 4: CHẠY THỬ NGHIỆM MÔ HÌNH A2C ĐÃ HUẤN LUYỆN
eval_env_a2c = TicTacToeDiscreteEnv(opponent_type="minimax")
loaded_a2c = A2C.load(a2c_save_path, env=eval_env_a2c)
evaluate_trained_agent(loaded_a2c, eval_env_a2c, num_games=10, is_continuous=False)

## 5. Mô Hình 4: SAC (Soft Actor-Critic - Maximum Entropy DRL)

- **Đặc tính**: Thuật toán Off-Policy Actor-Critic tối ưu hóa đồng thời phần thưởng và độ hỗn loạn (Entropy): 
  $$J(\pi) = \sum_{t} \mathbb{E} \left[ R(s_t, a_t) + \alpha \mathcal{H}(\pi(\cdot|s_t)) \right]$$
- **Áp dụng vào Tic-Tac-Toe**: Sử dụng môi trường `TicTacToeContinuousEnv`, xuất vector 9 chiều liên tục và chọn ô có độ ưu tiên cao nhất.

In [ ]:
# CELL 1: KHỞI TẠO MÔ HÌNH SAC (CODE GỐC FINRL SAC_PARAMS)
env_sac = DummyVecEnv([lambda: TicTacToeContinuousEnv(opponent_type="minimax")])

# Siêu tham số chuẩn từ finrl/config.py: SAC_PARAMS
SAC_KWARGS = {
    "batch_size": 64,
    "buffer_size": 20000,
    "learning_rate": 0.0003,
    "learning_starts": 150,
    "ent_coef": "auto_0.1",
    "gamma": 0.99,
    "verbose": 0,
}

model_sac = SAC(
    policy="MlpPolicy",
    env=env_sac,
    seed=42,
    policy_kwargs=dict(net_arch=dict(pi=[128, 64], qf=[128, 64])),
    **SAC_KWARGS
)

print("✅ Mô hình SAC đã được khởi tạo thành công với cấu hình FinRL!")
print(model_sac.policy)

In [ ]:
# CELL 2: HUẤN LUYỆN SAC VÀ LƯU CHECKPOINT
SAC_TIMESTEPS = 4000
sac_callback = FinRLGameMetricsCallback(eval_freq=100)

print(f"🚀 Bắt đầu huấn luyện SAC ({SAC_TIMESTEPS} timesteps)...")
start_time = time.time()
model_sac.learn(total_timesteps=SAC_TIMESTEPS, callback=sac_callback)
print(f"⏱️ Huấn luyện xong sau: {time.time() - start_time:.2f}s")

# Lưu Checkpoint vào thư mục yêu cầu
sac_save_path = os.path.join(CHECKPOINT_DIR, "sac_ttt.zip")
model_sac.save(sac_save_path)
print(f"💾 Đã lưu Checkpoint SAC thành công tại: {sac_save_path}")

In [ ]:
# CELL 3: VẼ ĐƯỜNG CONG HUẤN LUYỆN LOSS VÀ Q-VALUE CURVE CỦA SAC
plot_model_curves(sac_callback, model_name="SAC")

In [ ]:
# CELL 4: CHẠY THỬ NGHIỆM MÔ HÌNH SAC ĐÃ HUẤN LUYỆN
eval_env_sac = TicTacToeContinuousEnv(opponent_type="minimax")
loaded_sac = SAC.load(sac_save_path, env=eval_env_sac)
evaluate_trained_agent(loaded_sac, eval_env_sac, num_games=10, is_continuous=True)

## 6. Mô Hình 5: TD3 (Twin Delayed DDPG - Cải Tiến Overestimation Bias)

- **Đặc tính**: Nâng cấp vượt trội so với DDPG thông qua 3 cơ chế đột phá:
  1. **Twin Critics**: Sử dụng 2 mạng Critic song song và lấy giá trị cực tiểu $\min(Q_1, Q_2)$ để triệt tiêu việc thổi phồng giá trị Q.
  2. **Delayed Policy Updates**: Cập nhật mạng Policy chậm hơn mạng Critic.
  3. **Target Policy Smoothing**: Thêm nhiễu ngẫu nhiên vào hành động mục tiêu để làm mượt bề mặt giá trị.

In [ ]:
# CELL 1: KHỞI TẠO MÔ HÌNH TD3 (CODE GỐC FINRL TD3_PARAMS)
env_td3 = DummyVecEnv([lambda: TicTacToeContinuousEnv(opponent_type="minimax")])

# Thêm Action Noise từ FinRL
n_actions = env_td3.action_space.shape[-1]
action_noise = NormalActionNoise(mean=np.zeros(n_actions), sigma=0.1 * np.ones(n_actions))

# Siêu tham số chuẩn từ finrl/config.py: TD3_PARAMS
TD3_KWARGS = {
    "batch_size": 64,
    "buffer_size": 20000,
    "learning_rate": 0.0003,
    "learning_starts": 150,
    "gamma": 0.99,
    "verbose": 0,
}

model_td3 = TD3(
    policy="MlpPolicy",
    env=env_td3,
    action_noise=action_noise,
    seed=42,
    policy_kwargs=dict(net_arch=dict(pi=[128, 64], qf=[128, 64])),
    **TD3_KWARGS
)

print("✅ Mô hình TD3 đã được khởi tạo thành công với cấu hình FinRL!")
print(model_td3.policy)

In [ ]:
# CELL 2: HUẤN LUYỆN TD3 VÀ LƯU CHECKPOINT
TD3_TIMESTEPS = 4000
td3_callback = FinRLGameMetricsCallback(eval_freq=100)

print(f"🚀 Bắt đầu huấn luyện TD3 ({TD3_TIMESTEPS} timesteps)...")
start_time = time.time()
model_td3.learn(total_timesteps=TD3_TIMESTEPS, callback=td3_callback)
print(f"⏱️ Huấn luyện xong sau: {time.time() - start_time:.2f}s")

# Lưu Checkpoint vào thư mục yêu cầu
td3_save_path = os.path.join(CHECKPOINT_DIR, "td3_ttt.zip")
model_td3.save(td3_save_path)
print(f"💾 Đã lưu Checkpoint TD3 thành công tại: {td3_save_path}")

In [ ]:
# CELL 3: VẼ ĐƯỜNG CONG HUẤN LUYỆN LOSS VÀ Q-VALUE CURVE CỦA TD3
plot_model_curves(td3_callback, model_name="TD3")

In [ ]:
# CELL 4: CHẠY THỬ NGHIỆM MÔ HÌNH TD3 ĐÃ HUẤN LUYỆN
eval_env_td3 = TicTacToeContinuousEnv(opponent_type="minimax")
loaded_td3 = TD3.load(td3_save_path, env=eval_env_td3)
evaluate_trained_agent(loaded_td3, eval_env_td3, num_games=10, is_continuous=True)

## 7. Mô Hình 6: DDPG (Deep Deterministic Policy Gradient)

- **Đặc tính**: Mô hình Actor-Critic tất định kinh điển cho không gian hành động liên tục.
- **Thêm Nhiễu Khám Phá (Exploration Noise)**: Sử dụng nhiễu `OrnsteinUhlenbeckActionNoise` mô phỏng chuyển động ngẫu nhiên có tương quan thời gian như trong giao dịch tài chính FinRL.

In [ ]:
# CELL 1: KHỞI TẠO MÔ HÌNH DDPG (CODE GỐC FINRL DDPG_PARAMS)
env_ddpg = DummyVecEnv([lambda: TicTacToeContinuousEnv(opponent_type="minimax")])

# Thêm nhiễu Ornstein-Uhlenbeck Action Noise đặc trưng của FinRL DDPG
n_actions = env_ddpg.action_space.shape[-1]
ou_noise = OrnsteinUhlenbeckActionNoise(mean=np.zeros(n_actions), sigma=0.15 * np.ones(n_actions))

# Siêu tham số chuẩn từ finrl/config.py: DDPG_PARAMS
DDPG_KWARGS = {
    "batch_size": 64,
    "buffer_size": 20000,
    "learning_rate": 0.0005,
    "gamma": 0.99,
    "verbose": 0,
}

model_ddpg = DDPG(
    policy="MlpPolicy",
    env=env_ddpg,
    action_noise=ou_noise,
    seed=42,
    policy_kwargs=dict(net_arch=dict(pi=[128, 64], qf=[128, 64])),
    **DDPG_KWARGS
)

print("✅ Mô hình DDPG đã được khởi tạo thành công với cấu hình FinRL!")
print(model_ddpg.policy)

In [ ]:
# CELL 2: HUẤN LUYỆN DDPG VÀ LƯU CHECKPOINT
DDPG_TIMESTEPS = 4000
ddpg_callback = FinRLGameMetricsCallback(eval_freq=100)

print(f"🚀 Bắt đầu huấn luyện DDPG ({DDPG_TIMESTEPS} timesteps)...")
start_time = time.time()
model_ddpg.learn(total_timesteps=DDPG_TIMESTEPS, callback=ddpg_callback)
print(f"⏱️ Huấn luyện xong sau: {time.time() - start_time:.2f}s")

# Lưu Checkpoint vào thư mục yêu cầu
ddpg_save_path = os.path.join(CHECKPOINT_DIR, "ddpg_ttt.zip")
model_ddpg.save(ddpg_save_path)
print(f"💾 Đã lưu Checkpoint DDPG thành công tại: {ddpg_save_path}")

In [ ]:
# CELL 3: VẼ ĐƯỜNG CONG HUẤN LUYỆN LOSS VÀ Q-VALUE CURVE CỦA DDPG
plot_model_curves(ddpg_callback, model_name="DDPG")

In [ ]:
# CELL 4: CHẠY THỬ NGHIỆM MÔ HÌNH DDPG ĐÃ HUẤN LUYỆN
eval_env_ddpg = TicTacToeContinuousEnv(opponent_type="minimax")
loaded_ddpg = DDPG.load(ddpg_save_path, env=eval_env_ddpg)
evaluate_trained_agent(loaded_ddpg, eval_env_ddpg, num_games=10, is_continuous=True)

## 8. Bảng So Sánh Tổng Hợp Hiệu Năng 6 Mô Hình FinRL Trên Game Tic-Tac-Toe

Chạy đối đầu toàn diện 20 ván đối với từng mô hình đã nạp từ thư mục `practice/TicTacToe_framework/files/FinRL_Checkpoints` để tạo bảng so sánh khách quan:

In [ ]:
# CELL TỔNG HỢP SO SÁNH HIỆU NĂNG 6 MÔ HÌNH
models_dict = {
    "DQN": (DQN, os.path.join(CHECKPOINT_DIR, "dqn_ttt.zip"), False),
    "PPO": (PPO, os.path.join(CHECKPOINT_DIR, "ppo_ttt.zip"), False),
    "A2C": (A2C, os.path.join(CHECKPOINT_DIR, "a2c_ttt.zip"), False),
    "SAC": (SAC, os.path.join(CHECKPOINT_DIR, "sac_ttt.zip"), True),
    "TD3": (TD3, os.path.join(CHECKPOINT_DIR, "td3_ttt.zip"), True),
    "DDPG": (DDPG, os.path.join(CHECKPOINT_DIR, "ddpg_ttt.zip"), True),
}

print("=========================================================================================")
print(f"| {'MÔ HÌNH':<8} | {'LOẠI THUẬT TOÁN':<20} | {'KHÔNG GIAN':<12} | {'TỶ LỆ BẤT BẠI':<15} | {'TRẠNG THÁI':<15} |")
print("=========================================================================================")

benchmark_results = []
for name, (cls, path, is_cont) in models_dict.items():
    algo_type = "Value-Based" if name == "DQN" else ("On-Policy AC" if name in ["PPO", "A2C"] else "Off-Policy AC")
    space_type = "Continuous" if is_cont else "Discrete"
    
    if os.path.exists(path):
        test_env = TicTacToeContinuousEnv() if is_cont else TicTacToeDiscreteEnv()
        m = cls.load(path, env=test_env)
        wins, draws, total = 0, 0, 20
        for _ in range(total):
            obs, _ = test_env.reset()
            done = False
            while not done:
                act, _ = m.predict(obs, deterministic=True)
                obs, rew, term, trunc, info = test_env.step(act)
                done = term or trunc
            if info.get("result") == "win":
                wins += 1
            elif info.get("result") == "draw":
                draws += 1
        invincible_rate = (wins + draws) / total * 100
        status = "✅ Đã Kiểm Thử"
    else:
        invincible_rate = 0.0
        status = "⚠️ Chưa Train"
        
    print(f"| {name:<8} | {algo_type:<20} | {space_type:<12} | {invincible_rate:>13.1f}% | {status:<15} |")

print("=========================================================================================")

## 9. Tổng Kết & Nhận Xét Chuyên Sâu

1. **Mô hình rời rạc (DQN, PPO, A2C)**:
   - Phù hợp tự nhiên nhất với bản chất của bàn cờ Tic-Tac-Toe.
   - **DQN** cho đường cong học tập Q-value trực quan, thể hiện rõ quá trình tối ưu hóa phương trình Bellman.
   - **PPO** thể hiện độ ổn định cao nhất, không bị dao động chính sách lớn nhờ cơ chế Clip Surrogate.
2. **Mô hình liên tục (SAC, TD3, DDPG)**:
   - Khi áp dụng vào game rời rạc thông qua vector ưu tiên liên tục 9 chiều (`Box(9,)`), các mô hình này học cách gán điểm xác suất liên tục cho từng ô trống.
   - **SAC** đạt khả năng khám phá tốt nhờ Maximum Entropy, trong khi **TD3** vượt trội hơn DDPG nhờ cơ chế Twin Critics ngăn ngừa việc ước lượng sai giá trị Q.
3. **Ý nghĩa ứng dụng thực tiễn**:
   - Việc đóng gói chuẩn mực DRLAgent của FinRL giúp dễ dàng mở rộng sang các bài toán game bàn cờ phức tạp hơn (Connect Four, Gomoku) cũng như quay lại ứng dụng tối ưu hóa danh mục đầu tư tài chính trong FinRL.